In [1]:
!pip install prophet
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from datetime import datetime, timedelta

print("Libraries loaded. Ready to wash.")

Libraries loaded. Ready to wash.


In [2]:

dates = pd.date_range(start='2024-01-01', end='2025-12-31', freq='D')
df = pd.DataFrame({'ds': dates})

def generate_usage(row):
    # Base usage + Weekend spike + Seasonal Exam spike
    base = 20
    day_effect = 15 if row.dayofweek >= 5 else 0 # Weekends are busy
    exam_effect = 25 if row.month in [5, 12] else 0 # Finals month surge
    noise = np.random.normal(0, 5)
    return max(0, base + day_effect + exam_effect + noise)

df['y'] = df['ds'].apply(generate_usage)
df['day_of_week'] = df['ds'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['month'] = df['ds'].dt.month


def categorize(y):
    if y > 45: return 'Heavy'
    if y > 25: return 'Moderate'
    return 'Light'

df['usage_category'] = df['y'].apply(categorize)
df.head()

,ds,y,day_of_week,is_weekend,month,usage_category
0,2024-01-01,25.964375,0,0,1,Moderate
1,2024-01-02,27.332835,1,0,1,Moderate
2,2024-01-03,20.069769,2,0,1,Light
3,2024-01-04,22.099787,3,0,1,Light
4,2024-01-05,22.654813,4,0,1,Light


In [3]:

le = LabelEncoder()
X = df[['day_of_week', 'month', 'is_weekend']]
y_cat = le.fit_transform(df['usage_category'])

nb_model = GaussianNB()
nb_model.fit(X, y_cat)

m = Prophet(yearly_seasonality=True, weekly_seasonality=True)
m.fit(df[['ds', 'y']])

future = m.make_future_dataframe(periods=60)
forecast = m.predict(future)

print("Models trained successfully.")

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Models trained successfully.


In [4]:
def update_dashboard(days_ahead):
    target_date = datetime.now() + timedelta(days=days_ahead)


    pred = forecast[forecast['ds'].dt.date == target_date.date()]
    val = pred['yhat'].values[0] if not pred.empty else 0


    features = [[target_date.weekday(), target_date.month, 1 if target_date.weekday() >= 5 else 0]]
    cat_idx = nb_model.predict(features)[0]
    cat_name = le.inverse_common_classes_ = le.inverse_transform([cat_idx])[0]


    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = val,
        title = {'text': f"Predicted Loads for {target_date.strftime('%Y-%m-%d')}"},
        gauge = {'axis': {'range': [0, 80]},
                 'bar': {'color': "darkblue"},
                 'steps': [
                     {'range': [0, 25], 'color': "lightcyan"},
                     {'range': [25, 45], 'color': "royalblue"},
                     {'range': [45, 80], 'color': "midnightblue"}]}))

    print(f"Classification Logic: {cat_name} Usage Expected")
    fig.show()

slider = widgets.IntSlider(value=1, min=1, max=60, step=1, description='Days Ahead:')
widgets.interactive(update_dashboard, days_ahead=slider)

interactive(children=(IntSlider(value=1, description='Days Ahead:', max=60, min=1), Output()), _dom_classes=('…

In [5]:
def calculate_resources(predicted_load, category):
    # Logic: Heavy days need more buffer; 1 machine handles ~8 loads/day
    buffer = 1.2 if category == 'Heavy' else 1.05
    needed_machines = np.ceil((predicted_load * buffer) / 8)

    # Calculate expected wait time (Simulated M/M/1 Queue logic)
    # If load is 90% of capacity, wait times spike
    utilization = predicted_load / (needed_machines * 8)
    wait_time_mins = (utilization**2 / (1 - utilization)) * 30 if utilization < 1 else 120

    return int(needed_machines), round(min(wait_time_mins, 180), 1)

print("Resource logic defined.")

Resource logic defined.


In [6]:
def stress_test_dashboard(days_ahead, broken_machines):
    target_date = datetime.now() + timedelta(days=days_ahead)


    pred = forecast[forecast['ds'].dt.date == target_date.date()]
    val = pred['yhat'].values[0] if not pred.empty else 0


    features = [[target_date.weekday(), target_date.month, 1 if target_date.weekday() >= 5 else 0]]
    cat_idx = nb_model.predict(features)[0]
    cat_name = le.inverse_transform([cat_idx])[0]

    # Resource Calc
    req_machines, wait = calculate_resources(val, cat_name)
    available_machines = 10 - broken_machines # Assuming hostel has 10 machines total

    # Status Color
    status_color = "red" if available_machines < req_machines else "green"

    print(f"--- Report for {target_date.strftime('%A, %b %d')} ---")
    print(f"Predicted Category: {cat_name}")
    print(f"Estimated Loads: {round(val, 1)}")
    print(f"Machines Needed: {req_machines} | Machines Available: {available_machines}")
    print(f"Projected Peak Wait Time: {wait if available_machines >= req_machines else 'INFINITE (Bottleneck)'} mins")

    # Visual Alert
    fig = go.Figure(go.Indicator(
        mode = "number+delta",
        value = available_machines,
        delta = {'reference': req_machines, 'relative': False},
        title = {'text': "Machine Surplus/Deficit"},
        domain = {'x': [0, 1], 'y': [0, 1]}))
    fig.show()

# Interactive Sliders
widgets.interactive(stress_test_dashboard,
                   days_ahead=widgets.IntSlider(min=1, max=60, value=7),
                   broken_machines=widgets.IntSlider(min=0, max=5, value=0, description='Broken:'))

interactive(children=(IntSlider(value=7, description='days_ahead', max=60, min=1), IntSlider(value=0, descript…

In [7]:

merged = df.merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds')
merged['anomaly'] = (merged['y'] > merged['yhat_upper']) | (merged['y'] < merged['yhat_lower'])

anomalies = merged[merged['anomaly'] == True]
print(f"Detected {len(anomalies)} unusual laundry spikes in history.")
print(anomalies[['ds', 'y', 'yhat']].head())

Detected 137 unusual laundry spikes in history.
           ds          y       yhat
2  2024-01-03  20.069769  30.430103
11 2024-01-12  12.479726  21.094841
19 2024-01-20  40.417393  32.856436
23 2024-01-24  26.112668  18.757317
28 2024-01-29  29.215104  18.400062
